# 2 · Pragmatic Classification — Model Comparison

**Measuring Pragmatic Alignment in LLM-Based Agents**
University of Trier · NLP Master's Program · WS 2025/26

Can the four pragmatic dimensions (STANCE · ACTION · PERSONALNESS · POLITENESS)
be predicted automatically? This notebook compares lexical, frozen-embedding and
fine-tuned approaches on a single stratified split, and asks a second question
that the annotation design raises: **does the target tweet actually help a
classifier, or only the human annotator?**

| Step | What it does |
|------|-------------|
| 1 | Load the prepared dataset and encode labels |
| 2 | Build one stratified train/val/test split, reused by every model |
| 3 | Build three input representations from frozen SBERT |
| 4 | Majority-class and TF-IDF baselines |
| 5 | Representation ablation across three classical models |
| 6 | Fine-tune MiniLM-L6 and RoBERTa-base, both input modes |
| 7 | Comparative results |
| 8 | Serialise the critics used by the agentic pipeline |

Accuracy is reported alongside **macro-F1**. The labels are heavily skewed, so
accuracy on its own is close to uninformative — the majority-class baseline in
section 4 makes that concrete.

In [1]:
import os

# On macOS, xgboost and torch each bundle an OpenMP runtime; loading both
# unguarded crashes the interpreter. Pin the thread count before either is
# imported, and import xgboost first so its runtime is the one in use.
os.environ.setdefault("OMP_NUM_THREADS", "1")

from xgboost import XGBClassifier  # noqa: E402  (must precede torch)

import copy  # noqa: E402
import json  # noqa: E402
import random  # noqa: E402
import warnings  # noqa: E402
from pathlib import Path  # noqa: E402

import joblib  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import torch  # noqa: E402
from sklearn.dummy import DummyClassifier  # noqa: E402
from sklearn.feature_extraction.text import TfidfVectorizer  # noqa: E402
from sklearn.linear_model import LogisticRegression  # noqa: E402
from sklearn.metrics import accuracy_score, classification_report, f1_score  # noqa: E402
from sklearn.model_selection import train_test_split  # noqa: E402
from sklearn.preprocessing import LabelEncoder  # noqa: E402
from sklearn.svm import LinearSVC  # noqa: E402

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA = Path("../data/annotated_clean.csv")
OUT = Path("../notebooks")
LABEL_COLS = ["STANCE", "ACTION", "PERSONALNESS", "POLITENESS"]

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"seed={SEED}  device={DEVICE}")

seed=42  device=mps


---
## 1. Load data and encode labels

In [2]:
df = pd.read_csv(DATA)
print(f"rows: {len(df)}")

encoders, Y = {}, pd.DataFrame(index=df.index)
for col in LABEL_COLS:
    le = LabelEncoder().fit(df[col])
    encoders[col] = le
    Y[col] = le.transform(df[col])
    dist = "  ".join(f"{k} {v} ({v / len(df):.0%})" for k, v in df[col].value_counts().items())
    print(f"{col:13} {len(le.classes_)} classes — {dist}")

rows: 800
STANCE        3 classes — OPPOSE 360 (45%)  NEUTRAL 246 (31%)  SUPPORT 194 (24%)
ACTION        4 classes — STATEMENT 571 (71%)  QUESTION 119 (15%)  COMMAND 89 (11%)  REACTION 21 (3%)
PERSONALNESS  2 classes — GENERAL 674 (84%)  PERSONAL 126 (16%)
POLITENESS    3 classes — NORMAL 619 (77%)  RUDE 156 (20%)  POLITE 25 (3%)


---
## 2. One stratified split, shared by every model

The dataset is small and the labels are skewed, so an unstratified split can
leave a rare class absent from the test set. We stratify on the joint label
combination, collapsing combinations too rare to split into a single `RARE`
stratum.

The **validation** split chooses the stopping epoch when fine-tuning. The
**test** split is touched only for the final numbers reported below.

In [3]:
joint = df[LABEL_COLS].agg("|".join, axis=1)
strata = joint.where(joint.map(joint.value_counts()) >= 8, "RARE")
print(f"{strata.nunique()} strata (from {joint.nunique()} raw label combinations)")

idx_train, idx_temp = train_test_split(df.index, test_size=0.30, random_state=SEED, stratify=strata)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, random_state=SEED, stratify=strata[idx_temp])
print(f"train {len(idx_train)} · val {len(idx_val)} · test {len(idx_test)}")

print("\nClass counts per split (train/val/test):")
for col in LABEL_COLS:
    parts = [
        f"{cls} {(df.loc[idx_train, col] == cls).sum()}/{(df.loc[idx_val, col] == cls).sum()}/{(df.loc[idx_test, col] == cls).sum()}"
        for cls in encoders[col].classes_
    ]
    print(f"  {col:13} " + "  ".join(parts))

19 strata (from 38 raw label combinations)
train 560 · val 120 · test 120

Class counts per split (train/val/test):
  STANCE        NEUTRAL 172/36/38  OPPOSE 253/54/53  SUPPORT 135/30/29
  ACTION        COMMAND 60/16/13  QUESTION 83/19/17  REACTION 14/3/4  STATEMENT 403/82/86
  PERSONALNESS  GENERAL 468/102/104  PERSONAL 92/18/16
  POLITENESS    NORMAL 434/95/90  POLITE 16/3/6  RUDE 110/22/24


> Note how thin the rare classes are: `REACTION` and `POLITE` have only a
> handful of test instances each. Per-class scores for those two are dominated
> by sampling noise and should not be over-interpreted.

---
## 3. Three input representations

The annotation scheme labels a reply *in the context of* the tweet it answers.
That is necessary for a human annotator — stance is not judgeable otherwise.
Whether a classifier can exploit the same context is a separate question, so we
build three representations from the frozen SBERT encoder and test all three.

| Representation | Vector | Idea |
|---|---|---|
| `concat-string` | 384-d | embed `"tweet reply"` as a single string |
| `pair` | 1536-d | `[u, v, abs(u-v), u*v]` over separate tweet/reply embeddings |
| `reply-only` | 384-d | ignore the tweet entirely |

In [4]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)
enc_kw = dict(batch_size=64, show_progress_bar=False)

emb_concat = embedder.encode((df.target_tweet + " " + df.authentic_reply).tolist(), **enc_kw)
emb_tweet = embedder.encode(df.target_tweet.tolist(), **enc_kw)
emb_reply = embedder.encode(df.authentic_reply.tolist(), **enc_kw)
emb_pair = np.hstack([emb_tweet, emb_reply, np.abs(emb_tweet - emb_reply), emb_tweet * emb_reply])

REPRESENTATIONS = {"concat-string": emb_concat, "pair": emb_pair, "reply-only": emb_reply}
for name, mat in REPRESENTATIONS.items():
    print(f"{name:14} {mat.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

concat-string  (800, 384)
pair           (800, 1536)
reply-only     (800, 384)


---
## 4. Evaluation helper, and the two baselines

In [5]:
RESULTS = {}

def evaluate(name, preds, verbose=True):
    """Score per-dimension predictions on the test split and store the summary."""
    rows = []
    for col in LABEL_COLS:
        y_true, y_pred = Y.loc[idx_test, col].values, preds[col]
        rows.append({
            "Dimension": col,
            "Accuracy": accuracy_score(y_true, y_pred),
            "Macro-F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "Weighted-F1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        })
    table = pd.DataFrame(rows).set_index("Dimension")
    table.loc["MEAN"] = table.mean()
    RESULTS[name] = table
    if verbose:
        print(f"\n=== {name} ===")
        print(table.round(4).to_string())
    return table

def fit_predict(make_model, X):
    """Train one model per dimension on `X` and return test-set predictions."""
    preds = {}
    for col in LABEL_COLS:
        model = make_model()
        model.fit(X[idx_train], Y.loc[idx_train, col])
        preds[col] = model.predict(X[idx_test])
    return preds

**Majority-class baseline.** A classifier that ignores its input entirely and
always predicts the most frequent training class. Every accuracy figure below
has to be read against this number.

In [6]:
evaluate("Majority-class baseline", fit_predict(lambda: DummyClassifier(strategy="most_frequent"), emb_reply))


=== Majority-class baseline ===
              Accuracy  Macro-F1  Weighted-F1
Dimension                                    
STANCE          0.4417    0.2042       0.2706
ACTION          0.7167    0.2087       0.5984
PERSONALNESS    0.8667    0.4643       0.8048
POLITENESS      0.7500    0.2857       0.6429
MEAN            0.6938    0.2907       0.5792


,Accuracy,Macro-F1,Weighted-F1
Dimension,,,
STANCE,0.441667,0.204239,0.270617
ACTION,0.716667,0.208738,0.598382
PERSONALNESS,0.866667,0.464286,0.804762
POLITENESS,0.750000,0.285714,0.642857
MEAN,0.693750,0.290744,0.579154


**Lexical baseline.** TF-IDF n-grams with a linear SVM — can pragmatic function
be read off surface word choice alone?

In [7]:
tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1, 2), stop_words="english")
text_train = df.loc[idx_train, "target_tweet"] + " " + df.loc[idx_train, "authentic_reply"]
text_test = df.loc[idx_test, "target_tweet"] + " " + df.loc[idx_test, "authentic_reply"]
X_tfidf_train, X_tfidf_test = tfidf.fit_transform(text_train), tfidf.transform(text_test)

preds = {}
for col in LABEL_COLS:
    clf = LinearSVC(class_weight="balanced", random_state=SEED)
    clf.fit(X_tfidf_train, Y.loc[idx_train, col])
    preds[col] = clf.predict(X_tfidf_test)
evaluate("TF-IDF + LinearSVC", preds)


=== TF-IDF + LinearSVC ===
              Accuracy  Macro-F1  Weighted-F1
Dimension                                    
STANCE          0.4333    0.3869       0.4144
ACTION          0.6833    0.2441       0.6077
PERSONALNESS    0.8500    0.5496       0.8193
POLITENESS      0.7333    0.3425       0.6707
MEAN            0.6750    0.3808       0.6280


,Accuracy,Macro-F1,Weighted-F1
Dimension,,,
STANCE,0.433333,0.386853,0.414353
ACTION,0.683333,0.244082,0.607701
PERSONALNESS,0.850000,0.549625,0.819349
POLITENESS,0.733333,0.342530,0.670692
MEAN,0.675000,0.380772,0.628024


---
## 5. Representation ablation — does the target tweet help?

Three classical models times three representations, all on frozen SBERT
features and the same split.

In [8]:
CLASSICAL = {
    "LinearSVC": lambda: LinearSVC(class_weight="balanced", random_state=SEED),
    "LogReg": lambda: LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED),
    "XGBoost": lambda: XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8,
        colsample_bytree=0.8, eval_metric="mlogloss", tree_method="hist", random_state=SEED,
    ),
}

for rep_name, X in REPRESENTATIONS.items():
    for model_name, make in CLASSICAL.items():
        evaluate(f"{model_name} · {rep_name}", fit_predict(make, X), verbose=False)

for metric in ["Macro-F1", "Accuracy"]:
    table = pd.DataFrame({
        rep: {m: RESULTS[f"{m} · {rep}"].loc["MEAN", metric] for m in CLASSICAL}
        for rep in REPRESENTATIONS
    })
    print(f"Mean {metric} by model x representation:\n")
    print(table.round(4).to_string())
    print()

Mean Macro-F1 by model x representation:

           concat-string    pair  reply-only
LinearSVC         0.3974  0.4739      0.5173
LogReg            0.4318  0.5073      0.5145
XGBoost           0.3641  0.3330      0.3721

Mean Accuracy by model x representation:

           concat-string    pair  reply-only
LinearSVC         0.6271  0.6833      0.6688
LogReg            0.5854  0.6667      0.6375
XGBoost           0.6958  0.6938      0.6979



> **Finding.** For frozen embeddings the target tweet is not usable signal: the
> `reply-only` representation matches or beats both representations that include
> it, for every model. Concatenating tweet and reply into one string — the
> obvious first choice — is the *worst* option for the linear models, because it
> blurs two sentences into a single averaged vector. Structuring the pair
> explicitly (`pair`) recovers most of that loss.
>
> This does not invalidate annotating in context: the annotators needed the
> tweet to assign stance at all. The point is that 560 training examples are not
> enough to learn to *use* it as model input. Section 6 tests whether end-to-end
> fine-tuning changes that — it does not: reply-only is selected on validation for
> both fine-tuned models as well.

---
## 6. Fine-tuned transformers

A shared encoder with four classification heads, trained jointly. Two choices
matter relative to a naive setup:

1. **Validation-based epoch selection** — train for 10 epochs and keep the
   weights from the epoch with the best validation macro-F1, rather than
   whatever the final epoch happens to produce.
2. **Both input modes are tried** — true sentence-pair encoding
   (`tokenizer(tweet, reply)`, which gives the model real segment structure)
   and reply-only, mirroring the ablation in section 5.

Both MiniLM-L6 (23M parameters — the same encoder used frozen above) and
RoBERTa-base (125M) are fine-tuned, so the effect of model capacity is visible
separately from the effect of the training regime. Each is run with **both**
input modes, mirroring the ablation in section 5, and the mode reported in the
headline table is the one with the better *validation* score — the test split is
never used to make that choice.

In [9]:
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

MAX_LEN, BATCH, EPOCHS, LR = 128, 16, 10, 2e-5
NUM_CLASSES = [len(encoders[c].classes_) for c in LABEL_COLS]

class PragmaticDataset(Dataset):
    """Tokenised (tweet, reply) pairs, or replies alone when `mode='reply'`."""

    def __init__(self, idx, tokenizer, mode):
        self.tweets = df.loc[idx, "target_tweet"].tolist()
        self.replies = df.loc[idx, "authentic_reply"].tolist()
        self.labels = Y.loc[idx].values
        self.tokenizer, self.mode = tokenizer, mode

    def __len__(self):
        return len(self.replies)

    def __getitem__(self, i):
        kw = dict(truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt")
        enc = (self.tokenizer(self.tweets[i], self.replies[i], **kw) if self.mode == "pair"
               else self.tokenizer(self.replies[i], **kw))
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[i], dtype=torch.long)
        return item

class MultiHeadTransformer(torch.nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = torch.nn.Dropout(0.1)
        self.heads = torch.nn.ModuleList([torch.nn.Linear(hidden, n) for n in num_classes])

    def forward(self, input_ids, attention_mask, **kwargs):
        pooled = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        return [head(self.dropout(pooled)) for head in self.heads]

VAL_SCORES = {}

def finetune(label, model_name, mode="pair"):
    """Fine-tune end-to-end, keeping the epoch with the best validation macro-F1."""
    torch.manual_seed(SEED)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = MultiHeadTransformer(model_name, NUM_CLASSES).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"\n--- {label} ({model_name}, {n_params:.0f}M params, {mode} encoding) ---")

    loaders = {
        split: DataLoader(PragmaticDataset(idx, tokenizer, mode), batch_size=BATCH, shuffle=(split == "train"))
        for split, idx in [("train", idx_train), ("val", idx_val), ("test", idx_test)]
    }
    optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    loss_fn = torch.nn.CrossEntropyLoss()

    def predict(loader):
        model.eval()
        collected = [[] for _ in LABEL_COLS]
        with torch.no_grad():
            for batch in loader:
                batch.pop("labels")
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                for i, logit in enumerate(model(**batch)):
                    collected[i].append(logit.argmax(-1).cpu().numpy())
        return {c: np.concatenate(collected[i]) for i, c in enumerate(LABEL_COLS)}

    best_f1, best_state = -1.0, None
    for epoch in range(EPOCHS):
        model.train()
        total = 0.0
        for batch in loaders["train"]:
            labels = batch.pop("labels").to(DEVICE)
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optim.zero_grad()
            logits = model(**batch)
            loss = sum(loss_fn(logits[i], labels[:, i]) for i in range(len(LABEL_COLS)))
            loss.backward()
            optim.step()
            total += loss.item()

        val_preds = predict(loaders["val"])
        val_f1 = float(np.mean([
            f1_score(Y.loc[idx_val, c], val_preds[c], average="macro", zero_division=0)
            for c in LABEL_COLS
        ]))
        marker = ""
        if val_f1 > best_f1:
            best_f1, best_state, marker = val_f1, copy.deepcopy(model.state_dict()), "   <- best so far"
        print(f"  epoch {epoch + 1:2}/{EPOCHS}  train loss {total / len(loaders['train']):.4f}"
              f"  val macro-F1 {val_f1:.4f}{marker}")

    model.load_state_dict(best_state)
    print(f"  restored weights from best epoch (val macro-F1 {best_f1:.4f})")
    evaluate(label, predict(loaders["test"]))
    VAL_SCORES[label] = best_f1
    del model
    if DEVICE == "mps":
        torch.mps.empty_cache()

for base, name in [("MiniLM-L6", "sentence-transformers/all-MiniLM-L6-v2"),
                   ("RoBERTa-base", "roberta-base")]:
    for mode in ["pair", "reply-only"]:
        finetune(f"{base} FT · {mode}", name, mode="pair" if mode == "pair" else "reply")

print("\nValidation macro-F1 (the score the input mode is chosen on):")
for k, v in VAL_SCORES.items():
    print(f"  {k:28} {v:.4f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


--- MiniLM-L6 FT · pair (sentence-transformers/all-MiniLM-L6-v2, 23M params, pair encoding) ---


  epoch  1/10  train loss 3.3801  val macro-F1 0.3007   <- best so far


  epoch  2/10  train loss 2.9539  val macro-F1 0.3101   <- best so far


  epoch  3/10  train loss 2.8573  val macro-F1 0.3232   <- best so far


...

  epoch  7/10  train loss 0.7064  val macro-F1 0.6361   <- best so far


  epoch  8/10  train loss 0.4891  val macro-F1 0.6139


  epoch  9/10  train loss 0.3202  val macro-F1 0.6238


  epoch 10/10  train loss 0.2636  val macro-F1 0.6444   <- best so far
  restored weights from best epoch (val macro-F1 0.6444)



=== RoBERTa-base FT · reply-only ===
              Accuracy  Macro-F1  Weighted-F1
Dimension                                    
STANCE          0.5500    0.5274       0.5469
ACTION          0.8833    0.7573       0.8824
PERSONALNESS    0.8500    0.6755       0.8500
POLITENESS      0.7417    0.4410       0.7234
MEAN            0.7562    0.6003       0.7507

Validation macro-F1 (the score the input mode is chosen on):
  MiniLM-L6 FT · pair          0.4675
  MiniLM-L6 FT · reply-only    0.6188
  RoBERTa-base FT · pair       0.5559
  RoBERTa-base FT · reply-only 0.6444


---
## 7. Comparative results

In [10]:
summary = pd.DataFrame({
    name: {"Accuracy": t.loc["MEAN", "Accuracy"], "Macro-F1": t.loc["MEAN", "Macro-F1"]}
    for name, t in RESULTS.items()
}).T.sort_values("Macro-F1", ascending=False)
print("All configurations, mean across the four dimensions, best first:\n")
print(summary.round(4).to_string())

All configurations, mean across the four dimensions, best first:

                              Accuracy  Macro-F1
RoBERTa-base FT · reply-only    0.7562    0.6003
RoBERTa-base FT · pair          0.7312    0.5972
MiniLM-L6 FT · reply-only       0.7667    0.5752
LinearSVC · reply-only          0.6688    0.5173
LogReg · reply-only             0.6375    0.5145
LogReg · pair                   0.6667    0.5073
MiniLM-L6 FT · pair             0.7417    0.4936
LinearSVC · pair                0.6833    0.4739
LogReg · concat-string          0.5854    0.4318
LinearSVC · concat-string       0.6271    0.3974
TF-IDF + LinearSVC              0.6750    0.3808
XGBoost · reply-only            0.6979    0.3721
XGBoost · concat-string         0.6958    0.3641
XGBoost · pair                  0.6938    0.3330
Majority-class baseline         0.6938    0.2907


The headline comparison — the best configuration of each family:

In [11]:
# For each fine-tuned model the input mode is chosen by validation macro-F1,
# never by the test score.
best_mode = {
    base: max(["pair", "reply-only"], key=lambda m: VAL_SCORES[f"{base} FT · {m}"])
    for base in ["MiniLM-L6", "RoBERTa-base"]
}
print("Input mode selected on validation:", best_mode, "\n")

HEADLINE = [
    "Majority-class baseline",
    "TF-IDF + LinearSVC",
    "XGBoost · reply-only",
    "LinearSVC · reply-only",
    "LogReg · reply-only",
    f"MiniLM-L6 FT · {best_mode['MiniLM-L6']}",
    f"RoBERTa-base FT · {best_mode['RoBERTa-base']}",
]
print(summary.loc[HEADLINE].round(4).to_string())

print("\n\nPer-dimension macro-F1:\n")
print(pd.DataFrame({n: RESULTS[n]["Macro-F1"].drop("MEAN") for n in HEADLINE}).round(3).to_string())

Input mode selected on validation: {'MiniLM-L6': 'reply-only', 'RoBERTa-base': 'reply-only'} 

                              Accuracy  Macro-F1
Majority-class baseline         0.6938    0.2907
TF-IDF + LinearSVC              0.6750    0.3808
XGBoost · reply-only            0.6979    0.3721
LinearSVC · reply-only          0.6688    0.5173
LogReg · reply-only             0.6375    0.5145
MiniLM-L6 FT · reply-only       0.7667    0.5752
RoBERTa-base FT · reply-only    0.7562    0.6003


Per-dimension macro-F1:

              Majority-class baseline  TF-IDF + LinearSVC  XGBoost · reply-only  LinearSVC · reply-only  LogReg · reply-only  MiniLM-L6 FT · reply-only  RoBERTa-base FT · reply-only
Dimension                                                                                                                                                                            
STANCE                          0.204               0.387                 0.422                   0.450                0.4

---
## 8. Serialise the critics used by the agentic pipeline

The agentic pipeline (notebook 4) scores a candidate reply by asking a critic
for the probability it assigns to the intended class, so the critic must expose
calibrated per-class probabilities. The critics are therefore logistic-regression
heads on the frozen `reply-only` representation — the strongest probabilistic
classical configuration in the ablation above, and the appropriate choice since
the pipeline scores generated replies for which no gold "target tweet pairing"
context is being learned.

The XGBoost critics are written as well, so the earlier version of the pipeline
still loads, but note from section 5 that XGBoost is the weakest of the three.

In [12]:
critics = {}
for col in LABEL_COLS:
    clf = LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED)
    clf.fit(emb_reply[idx_train], Y.loc[idx_train, col])
    critics[col] = clf
joblib.dump(critics, OUT / "critics_logreg.pkl")
print(f"logreg critics -> critics_logreg.pkl "
      f"(mean macro-F1 {RESULTS['LogReg · reply-only'].loc['MEAN', 'Macro-F1']:.4f})")

for i, col in enumerate(LABEL_COLS):
    clf = CLASSICAL["XGBoost"]()
    clf.fit(emb_reply[idx_train], Y.loc[idx_train, col])
    clf.save_model(OUT / f"xgb_output_{i}.json")
    print(f"{col:13}  -> xgb_output_{i}.json  ({len(encoders[col].classes_)} classes)")

joblib.dump(encoders, OUT / "label_encoders.pkl")
print("encoders       -> label_encoders.pkl")

(OUT / "test_split_indices.json").write_text(json.dumps({
    "seed": SEED,
    "train": [int(i) for i in idx_train],
    "val": [int(i) for i in idx_val],
    "test": [int(i) for i in idx_test],
}))
print("split indices  -> test_split_indices.json")

logreg critics -> critics_logreg.pkl (mean macro-F1 0.5145)


STANCE         -> xgb_output_0.json  (3 classes)


ACTION         -> xgb_output_1.json  (4 classes)


PERSONALNESS   -> xgb_output_2.json  (2 classes)


POLITENESS     -> xgb_output_3.json  (3 classes)
encoders       -> label_encoders.pkl
split indices  -> test_split_indices.json


> **Reproducibility.** Everything except the RoBERTa fine-tune is exactly
> reproducible. RoBERTa's backward pass over its embedding table uses
> non-deterministic atomic accumulation on Apple Silicon (MPS), so its figures
> shift by roughly ±0.03 between runs; the ranking of the models is stable. All
> numbers quoted in the report are the ones stored in this notebook's outputs.